In [1]:
# imports that the notebook needs
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
# Loading the data and some preprocessing:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
           'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
           'hours-per-week', 'native-country', 'income']
df = pd.read_csv(url, names=columns, skipinitialspace=True).sample(8500, random_state=499) # I sampled the data so that it can be used in <1h

# Separate features and target (and drop some problematic variables)
X_raw = df.drop(['income', 'native-country', 'occupation'], axis=1)
y_raw = LabelEncoder().fit_transform(df['income'])

# We explicitly define which columns get scaled vs. which get one-hot encoded
numeric_features = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = ['workclass', 'marital-status', 'relationship', 'race', 'sex']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features), # Standardize to mean=0, std=1
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_features) #one hot encode categorical data
    ])

X_processed = pd.DataFrame(preprocessor.fit_transform(X_raw).astype(np.float32))
y_processed = pd.DataFrame(y_raw.astype(np.float32))

# split off 500 rows for testing
X_train_all, X_test, y_train_all, y_test = train_test_split(
    X_processed, y_processed, test_size=500, random_state=499
)

# # just resetting the indexes
# for df in [X_train_all, X_test, y_train_all, y_test]:
#     df.reset_index(drop=True, inplace=True)

print(X_train_all.head())

            0         1         2         3         4    5    6    7    8   \
3665  0.328328  1.148481 -0.141805 -0.210031 -0.046562  0.0  0.0  0.0  0.0   
288  -1.430200 -0.411207 -0.141805 -0.210031 -0.046562  0.0  0.0  0.0  0.0   
7215 -0.111304 -0.411207 -0.141805 -0.210031  3.534791  0.0  0.0  0.0  0.0   
1358  0.474871 -0.411207 -0.141805 -0.210031 -0.046562  0.0  0.0  0.0  0.0   
1836  0.914503  1.148481 -0.141805 -0.210031 -0.046562  0.0  0.0  0.0  0.0   

       9   ...   24   25   26   27   28   29   30   31   32   33  
3665  1.0  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  1.0  
288   1.0  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  1.0  0.0  
7215  0.0  ...  1.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  1.0  
1358  1.0  ...  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  1.0  
1836  1.0  ...  0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0  1.0  0.0  

[5 rows x 34 columns]


In [3]:
# Defining two architectures
class SmallNet(nn.Module):
    def __init__(self, input_dim):
        super(SmallNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 16)
        self.ReLU = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.ReLU(x)
        x = self.fc2(x)
        return self.sigmoid(x)

class LargeNet(nn.Module):
    def __init__(self, input_dim):
        super(LargeNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.ReLU = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.ReLU(x)
        x = self.fc2(x)
        x = self.ReLU(x)
        x = self.fc3(x)
        return self.sigmoid(x)

In [4]:
# The cross-validation loop

# this is a variable that allows us to view training loss for epoch X
print_training = True

# convert the data to Torch tensors and split the data into 5 folds
X_tensor = torch.tensor(X_train_all.values, dtype=torch.float32)
y_tensor = torch.tensor(y_train_all.values, dtype=torch.float32)
kf = KFold(n_splits=5, shuffle=True, random_state=499)

# create dictionaries to save the results
training_loss = {"SmallNet": [], "LargeNet": []}
train_accuracy = {"SmallNet": [], "LargeNet": []}
val_loss = {"SmallNet": [], "LargeNet": []}
val_accuracy = {"SmallNet": [], "LargeNet": []}

epochs = 10_000

for fold, (train_idx, val_idx) in enumerate(kf.split(X_tensor)):

    # Slice the tensors using indices
    X_train, y_train = X_tensor[train_idx], y_tensor[train_idx]
    X_val, y_val = X_tensor[val_idx], y_tensor[val_idx]

    # Initialize the models we defined above
    model_small = SmallNet(input_dim=X_train.shape[1])
    model_large = LargeNet(input_dim=X_train.shape[1])

    # optimizers and loss
    optimizer_small = torch.optim.SGD(model_small.parameters(), lr=0.01)
    optimizer_large = torch.optim.SGD(model_large.parameters(), lr=0.01)
    criterion = nn.BCELoss()

    print('--------------------------')
    print(f'Fold {fold+1}')
    print()

    for epoch in range(epochs):

        # Forward propagation
        outputs_small = model_small(X_train)
        outputs_large = model_large(X_train)

        # Calculate Training Loss and save
        loss_small = criterion(outputs_small, y_train)
        loss_large = criterion(outputs_large, y_train)

        # Backpropagation and Optimization
        optimizer_small.zero_grad()
        optimizer_large.zero_grad()
        loss_small.backward()
        loss_large.backward()
        optimizer_small.step()
        optimizer_large.step()

        if print_training:
            if (epoch + 1) % 1000 == 0:
                with torch.no_grad():
                    predictions_train_small = (outputs_small > 0.5).float()
                    predictions_train_large = (outputs_large > 0.5).float()
                    accuracy_train_small = (predictions_train_small == y_train).float().mean()
                    accuracy_train_large = (predictions_train_large == y_train).float().mean()

                    print(f'Epoch {epoch+1}')
                    print(f'Small model: Train Loss = {loss_small.item():.4f}, Train Acc = {accuracy_train_small.item():.2f}')
                    print(f'Large model: Train Loss = {loss_large.item():.4f}, Train Acc = {accuracy_train_large.item():.2f}')
                    print()

    print('Training Complete.')
    print()

    # Save final training loss and accuracy
    with torch.no_grad():
        training_loss["SmallNet"].append(loss_small.item())
        training_loss["LargeNet"].append(loss_large.item())
        train_accuracy["SmallNet"].append(accuracy_train_small.item())
        train_accuracy["LargeNet"].append(accuracy_train_large.item())

        # Calculate and Save validation loss and accuracy
        outputs_small = model_small(X_val)
        outputs_large = model_large(X_val)
        loss_val_small = criterion(outputs_small, y_val)
        loss_val_large = criterion(outputs_large, y_val)
        val_loss["SmallNet"].append(loss_val_small.item())
        val_loss["LargeNet"].append(loss_val_large.item())

        predictions_val_small = (outputs_small > 0.5).float()
        predictions_val_large = (outputs_large > 0.5).float()
        accuracy_val_small = (predictions_val_small == y_val).float().mean()
        accuracy_val_large = (predictions_val_large == y_val).float().mean()
        val_accuracy["SmallNet"].append(accuracy_val_small.item())
        val_accuracy["LargeNet"].append(accuracy_val_large.item())

        print(f'Final Validation Scores:')
        print(f'Small model: Validation Acc = {accuracy_val_small.item():.2f}')
        print(f'Large model: Validation Acc = {accuracy_val_large.item():.2f}')


/Users/itzjuztmya/miniconda3/envs/school/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--------------------------
Fold 1

Epoch 1000
Small model: Train Loss = 0.3663, Train Acc = 0.83
Large model: Train Loss = 0.3643, Train Acc = 0.83

Epoch 2000
Small model: Train Loss = 0.3314, Train Acc = 0.85
Large model: Train Loss = 0.3244, Train Acc = 0.85

Epoch 3000
Small model: Train Loss = 0.3205, Train Acc = 0.85
Large model: Train Loss = 0.3131, Train Acc = 0.85

Epoch 4000
Small model: Train Loss = 0.3147, Train Acc = 0.85
Large model: Train Loss = 0.3070, Train Acc = 0.86

Epoch 5000
Small model: Train Loss = 0.3111, Train Acc = 0.85
Large model: Train Loss = 0.3030, Train Acc = 0.86

Epoch 6000
Small model: Train Loss = 0.3087, Train Acc = 0.85
Large model: Train Loss = 0.3006, Train Acc = 0.86

Epoch 7000
Small model: Train Loss = 0.3067, Train Acc = 0.86
Large model: Train Loss = 0.2989, Train Acc = 0.86

Epoch 8000
Small model: Train Loss = 0.3052, Train Acc = 0.86
Large model: Train Loss = 0.2976, Train Acc = 0.86

Epoch 9000
Small model: Train Loss = 0.3040, Train Ac

In [5]:
# print all cross-validation results:
print(f"FINAL RESULTS:")
print()
print(f"SmallNet:")
print("Val Accuracy list: "+(", ".join(f"{x:.4f}" for x in val_accuracy['SmallNet'])))
print(f"Mean: {np.mean(val_accuracy['SmallNet']):.4f} (+/- {np.std(val_accuracy['SmallNet']):.4f})")
print()
print(f"LargeNet:")
print("Val Accuracy list: "+(", ".join(f"{x:.4f}" for x in val_accuracy['LargeNet'])))
print(f"Mean: {np.mean(val_accuracy['LargeNet']):.4f} (+/- {np.std(val_accuracy['LargeNet']):.4f})")

FINAL RESULTS:

SmallNet:
Val Accuracy list: 0.8350, 0.8669, 0.8587, 0.8444, 0.8506
Mean: 0.8511 (+/- 0.0111)

LargeNet:
Val Accuracy list: 0.8406, 0.8644, 0.8512, 0.8438, 0.8481
Mean: 0.8496 (+/- 0.0082)


In [6]:
# calculate a test accuracy
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

with torch.no_grad():
    outputs_test_small = model_small(X_test_tensor)
    outputs_test_large = model_large(X_test_tensor)
    predictions_test_small = (outputs_test_small > 0.5).float()
    predictions_test_large = (outputs_test_large > 0.5).float()
    accuracy_test_small = (predictions_test_small == y_test_tensor).float().mean()
    accuracy_test_large = (predictions_test_large == y_test_tensor).float().mean()
    print(f'SmallNet Test Accuracy: {accuracy_test_small.item():.4f}')
    print(f'LargeNet Test Accuracy: {accuracy_test_large.item():.4f}')

SmallNet Test Accuracy: 0.8640
LargeNet Test Accuracy: 0.8720
